In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Load dữ liệu thô
try:
    sales_df = pd.read_csv(r"C:\Users\Dell\Downloads\train_data_forecast.csv")
    products_df = pd.read_csv(r"C:\Users\Dell\Downloads\processed_products_VN_final.csv")
    interactions_df = pd.read_csv(r"C:\Users\Dell\Downloads\fact_user_interactions.csv")
    price_history_df = pd.read_csv(r"C:\Users\Dell\Downloads\fact_price_history_full.csv")

    print(" Đã load thành công 4 bộ dữ liệu!")
    print(f"- Sales Data: {sales_df.shape}")
    print(f"- Products Data: {products_df.shape}")
    print(f"- Interactions Data: {interactions_df.shape}")
    print(f"- Price History: {price_history_df.shape}")

except FileNotFoundError as e:
    print(f" LỖI: Không tìm thấy file. Chi tiết: {e}")

 Đã load thành công 4 bộ dữ liệu!
- Sales Data: (5672, 8)
- Products Data: (50, 13)
- Interactions Data: (24430, 6)
- Price History: (650, 6)


# 1. Module kiểm tra Cấu trúc (Completeness & Uniqueness)

In [2]:
def check_completeness(df):
    """
    Thang đo 1: COMPLETENESS (Tính đầy đủ)
    Kiểm tra các giá trị bị thiếu (Null/NaN) trong dataframe.
    """
    missing_count = df.isnull().sum().sum()
    missing_pct = (missing_count / (df.shape[0] * df.shape[1])) * 100
    
    print(f"1. Completeness (Độ đầy đủ):")
    print(f"   - Tổng giá trị thiếu: {missing_count} ({missing_pct:.2f}%)")
    
    if missing_count > 0:
        # Liệt kê các cột bị thiếu
        cols_missing = df.columns[df.isnull().any()].tolist()
        print(f"   - Các cột chứa Null: {cols_missing}")
    else:
        print("   - Dữ liệu đầy đủ 100%.")

def check_uniqueness(df, unique_cols=None):
    """
    Thang đo 2: UNIQUENESS (Tính duy nhất)
    Kiểm tra các dòng trùng lặp và trùng lặp trên khóa chính (Primary Key).
    """
    duplicates = df.duplicated().sum()
    print(f"2. Uniqueness (Độ duy nhất):")
    print(f"   - Số dòng trùng lặp hoàn toàn (Exact duplicates): {duplicates}")
    
    if unique_cols:
        # Chỉ kiểm tra nếu các cột key tồn tại trong df
        exist_cols = [c for c in unique_cols if c in df.columns]
        if exist_cols:
            key_dupes = df.duplicated(subset=exist_cols).sum()
            print(f"   - Trùng lặp trên Key {exist_cols}: {key_dupes}")
            if key_dupes > 0:
                print("     Cảnh báo: Khóa chính không duy nhất!")

# 2. Module kiểm tra Thời gian (Timeliness)

In [3]:
def check_timeliness(df, date_col):
    """
    Thang đo 3: TIMELINESS (Tính kịp thời)
    Kiểm tra định dạng thời gian và độ phủ (khoảng thời gian) của dữ liệu.
    """
    print(f"3. Timeliness (Tính kịp thời):")
    
    if date_col and date_col in df.columns:
        # Cố gắng chuyển đổi sang datetime nếu chưa phải
        if not pd.api.types.is_datetime64_any_dtype(df[date_col]):
             df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        
        # Tính toán khoảng thời gian
        min_date = df[date_col].min()
        max_date = df[date_col].max()
        
        if pd.notnull(min_date) and pd.notnull(max_date):
            days_diff = (max_date - min_date).days
            print(f"   - Dữ liệu từ: {min_date}")
            print(f"   - Đến ngày:   {max_date}")
            print(f"   - Độ phủ:     {days_diff} ngày")
        else:
            print("   - Cảnh báo: Cột thời gian chứa toàn giá trị lỗi (NaT).")
    else:
        print("   - Không áp dụng (Không tìm thấy cột thời gian).")

# 3. Module kiểm tra Logic nghiệp vụ (Validity)

In [4]:

def check_validity(df):
    """
    Thang đo 4: VALIDITY (Tính hợp lệ)
    Kiểm tra các quy tắc nghiệp vụ (Business Rules). 
    Ví dụ: Giá bán, Số lượng, Doanh thu không được phép âm.
    """
    print(f"4. Validity (Tính hợp lệ):")
    invalid_issues = []
    
    # Lấy các cột số
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    # Từ khóa để nhận diện các cột không được âm
    target_keywords = ['price', 'qty', 'quantity', 'sales', 'revenue', 'cost']
    
    for col in numeric_cols:
        # Chỉ kiểm tra các cột có tên chứa từ khóa
        if any(kw in col.lower() for kw in target_keywords):
            neg_count = (df[col] < 0).sum()
            if neg_count > 0:
                invalid_issues.append(f"Cột '{col}' có {neg_count} giá trị âm")
    
    if invalid_issues:
        print(f"   - PHÁT HIỆN LỖI LOGIC: {', '.join(invalid_issues)}")
    else:
        print("   - Dữ liệu số học hợp lệ (Không có số âm bất thường).")

# 4. Hàm Wrapper tổng hợp

In [5]:
# Hàm tổng hợp để gọi 4 thang đo chất lượng cho 1 bảng dữ liệu
def assess_data_quality_wrapper(df, df_name, unique_cols=None, date_col=None):
    print(f"\n{'='*20} ĐÁNH GIÁ: {df_name} {'='*20}")
    
    check_completeness(df)
    print("-" * 10)
    check_uniqueness(df, unique_cols)
    print("-" * 10)
    check_timeliness(df, date_col)
    print("-" * 10)
    check_validity(df)
    
    print("=" * 60)

# 5. Module kiểm tra Nhất quán liên bảng (Consistency)

In [6]:

def check_consistency(sales_df, products_df, price_history_df, interactions_df):
    """
    Thang đo 5: CONSISTENCY (Tính nhất quán)
    Kiểm tra toàn vẹn dữ liệu (Referential Integrity) giữa các bảng.
    """
    print(f"\n{'='*20} 5. CONSISTENCY CHECK (LIÊN BẢNG) {'='*20}")
    
    # Lấy tập hợp Product ID từ các bảng
    # Lưu ý: Tùy file mà tên cột có thể là 'Product ID' hoặc 'Product_ID'
    
    # Helper để lấy cột ID an toàn
    def get_ids(df, possible_names):
        for name in possible_names:
            if name in df.columns:
                return set(df[name].unique())
        return set()

    sales_pids = get_ids(sales_df, ['Product ID', 'Product_ID'])
    prod_pids = get_ids(products_df, ['Product ID', 'Product_ID'])
    inter_pids = get_ids(interactions_df, ['Product ID', 'Product_ID'])
    price_pids = get_ids(price_history_df, ['Product ID', 'Product_ID'])
    
    # 1. Kiểm tra Sales vs Products
    # Logic: Có bán hàng thì phải có thông tin sản phẩm
    orphan_sales = sales_pids - prod_pids
    if orphan_sales:
        print(f" CẢNH BÁO Sales-Products: Có {len(orphan_sales)} mã SP trong Sales nhưng KHÔNG CÓ trong Products.")
    else:
        print(" Sales -> Products: Nhất quán (Integrity OK).")

    # 2. Kiểm tra Interactions vs Products
    # Logic: Có tương tác thì phải có thông tin sản phẩm
    orphan_inter = inter_pids - prod_pids
    if orphan_inter:
         print(f" CẢNH BÁO Inter-Products: Có {len(orphan_inter)} mã SP trong Interactions nhưng KHÔNG CÓ trong Products.")
    else:
         print(" Interactions -> Products: Nhất quán (Integrity OK).")

    # 3. Kiểm tra Price History vs Products
    orphan_price = price_pids - prod_pids
    if orphan_price:
         print(f" CẢNH BÁO Price-Products: Có {len(orphan_price)} mã SP trong Price History nhưng KHÔNG CÓ trong Products.")
    else:
         print(" Price History -> Products: Nhất quán (Integrity OK).")
         
    print("=" * 60)

# 6. Thực thi đánh giá (Execution)

In [7]:
print(">>> BẮT ĐẦU QUÁ TRÌNH AUDIT DỮ LIỆU SENTIO <<<\n")

# 1. Đánh giá bảng Sản Phẩm
assess_data_quality_wrapper(products_df, "Dataset Sản Phẩm (Products)", unique_cols=['Product ID'])

# 2. Đánh giá bảng Doanh Số
# Lưu ý: Cột thời gian trong file Sales của cậu là 'Sales_Date'
assess_data_quality_wrapper(sales_df, "Dataset Doanh Số (Sales)", date_col='Sales_Date')

# 3. Đánh giá bảng Lịch sử Giá
# Lưu ý: Cột thời gian bắt đầu giá là 'Start_Date'
assess_data_quality_wrapper(price_history_df, "Dataset Lịch Sử Giá (Price)", date_col='Start_Date')

# 4. Đánh giá bảng Tương tác
# Lưu ý: Cột thời gian tương tác là 'Timestamp'
assess_data_quality_wrapper(interactions_df, "Dataset Tương Tác (Interactions)", date_col='Timestamp')

# 5. Kiểm tra tính nhất quán giữa các bảng
check_consistency(sales_df, products_df, price_history_df, interactions_df)

>>> BẮT ĐẦU QUÁ TRÌNH AUDIT DỮ LIỆU SENTIO <<<


==================== ĐÁNH GIÁ: Dataset Sản Phẩm (Products) ====================
1. Completeness (Độ đầy đủ):
   - Tổng giá trị thiếu: 153 (23.54%)
   - Các cột chứa Null: ['Color', 'image_url', 'model_3d_url', 'Is_Demo']
----------
2. Uniqueness (Độ duy nhất):
   - Số dòng trùng lặp hoàn toàn (Exact duplicates): 0
   - Trùng lặp trên Key ['Product ID']: 0
----------
3. Timeliness (Tính kịp thời):
   - Không áp dụng (Không tìm thấy cột thời gian).
----------
4. Validity (Tính hợp lệ):
   - Dữ liệu số học hợp lệ (Không có số âm bất thường).

==================== ĐÁNH GIÁ: Dataset Doanh Số (Sales) ====================
1. Completeness (Độ đầy đủ):
   - Tổng giá trị thiếu: 0 (0.00%)
   - Dữ liệu đầy đủ 100%.
----------
2. Uniqueness (Độ duy nhất):
   - Số dòng trùng lặp hoàn toàn (Exact duplicates): 0
----------
3. Timeliness (Tính kịp thời):
   - Dữ liệu từ: 2023-01-01 00:00:00
   - Đến ngày:   2023-03-21 00:00:00
   - Độ phủ:     79 ngày
--